## Plot fraction of individuals with low neutralization titers by strain

In [1]:
# Import packages
import os
import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gmean
from IPython.display import IFrame
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# Ignore error message from Altair about large dataframes
_ = alt.data_transformers.disable_max_rows()

# Basic color palette
color_palette = [
    '#345995', #blue
    '#03cea4', #teal
    '#ca1551', #red
    '#eac435', #yellow
               ]

In [2]:
# Define inputs
resultsdir = '../results'
os.makedirs(resultsdir, exist_ok = True)

# Define titers
titers = pd.read_csv(
    '../../../results/aggregated_titers/titers_EllebedyVaccineCohort.csv'
)

titers = titers.replace({'mRNA_': 'mRNA-1010_'})
titers['serum_new'] = titers['serum'].str.replace('mRNA_', 'mRNA-1010_', regex=False)
titers = titers.drop('serum', axis=1).rename(columns={'serum_new': 'serum'})


titers = titers.assign(
    vaccine = lambda x: x['serum'].str.split('_').str[0],
    participant = lambda x: x['serum'].str.split('_').str[1],
    timepoint = lambda x: x['serum'].str.split('_').str[2],
    subtype = lambda x: x['virus'].str.split('_').str[-1]
)

# Add another timepoint that includes 121 and 181 in teh same row
# Filter rows where timepoint is 121 or 181
late_tp_df = titers[titers['timepoint'].isin(['d121', 'd181'])].copy()
# Set new combined timepoint value
late_tp_df['timepoint'] = 'd121 or d181'
# Append to original DataFrame
titers = pd.concat([titers, late_tp_df], ignore_index=True)

# Calculate number of unique participants per vaccine
n_participants = titers.groupby('vaccine')['participant'].nunique()

# Create a mapping dictionary
vaccine_pretty = {v: f"{v} (n={n_participants[v]})" for v in n_participants.index}

# Map to a new pretty column
titers['vaccine_pretty'] = titers['vaccine'].map(vaccine_pretty)
titers['vaccine'] = titers['vaccine'].map(vaccine_pretty)

titers.serum.unique()

array(['Fluarix_397-001_d0', 'Fluarix_397-001_d181',
       'Fluarix_397-001_d29', 'Fluarix_397-002_d0',
       'Fluarix_397-002_d181', 'Fluarix_397-002_d29',
       'Fluarix_397-005_d0', 'Fluarix_397-005_d181',
       'Fluarix_397-005_d29', 'Fluarix_397-007_d0',
       'Fluarix_397-007_d121', 'Fluarix_397-007_d29',
       'Fluarix_397-009_d0', 'Fluarix_397-009_d121',
       'Fluarix_397-009_d29', 'Fluarix_397-011_d0',
       'Fluarix_397-011_d121', 'Fluarix_397-011_d29',
       'Fluarix_397-016_d0', 'Fluarix_397-016_d181',
       'Fluarix_397-016_d29', 'Fluarix_397-018_d0',
       'Fluarix_397-018_d121', 'Fluarix_397-018_d29',
       'Fluarix_397-020_d0', 'Fluarix_397-020_d181',
       'Fluarix_397-020_d29', 'Fluarix_397-022_d0',
       'Fluarix_397-022_d181', 'Fluarix_397-022_d29',
       'Fluarix_397-024_d0', 'Fluarix_397-024_d181',
       'Fluarix_397-024_d29', 'Fluarix_397-026_d0',
       'Fluarix_397-026_d181', 'Fluarix_397-026_d29',
       'Fluarix_397-027_d0', 'Fluarix_397-027_

In [3]:
# Define virus order
viral_plot_order = pd.read_csv('../../../data/H3_H1_ellebedy_library_strain_order.csv')
virus_order = [v for v in viral_plot_order.strain]

## Plot neutralization titers

In [4]:
def plot_titers_vaccination_cohorts(data, sort_order, _range = [30, 20000], title=None):
    # Make plot with all individuals and median dots
    color_scheme = alt.Color('timepoint',
                             sort=['d0', 'd29', 'd121 or d181']
                            ).scale(range=color_palette)
    titer_range = _range
    titleFontSize=18
    labelFontSize=18
    lineOpacity = 0.2
    lineSize = 2.8
    markerOpacity = 0.8
    markerSize = 160

    band = (alt.Chart(data)
            .mark_errorband(extent='iqr', opacity=0.4)
            .encode(alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right'),             
                          sort = virus_order),
                    alt.Y('titer', 
                          scale =alt.Scale(type='log',domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title="neutralization titer")),
                color = color_scheme,)
           ) 
    
    points = (alt.Chart(data)
              .mark_point(size = markerSize, stroke = 'black', strokeWidth = 2.2, filled=True,  opacity=markerOpacity)
              .encode(alt.X('virus', sort = virus_order),
                      alt.Y('median(titer)'),
                      color = color_scheme,))
        
    layered = (alt.layer(band, points)
               .facet(row = alt.Row('vaccine:N', sort=sort_order),
                      # column = alt.Column('timepoint', sort=['d0', 'd29', 'd121']),
                      config = alt.Config(legend = alt.LegendConfig(titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                                                    strokeColor='gray',padding=10,cornerRadius=10,
                                                    labelLimit = 1000 # Let legend labels be as long as they want
                                                     )))
               .properties(title=title)
               .configure_header(title=None,
                                  labelFontSize=labelFontSize,labelFontWeight='bold',
                                  labelOrient='right', 
                                )
               .configure_title(align='center', anchor='middle', fontSize=titleFontSize, fontWeight='bold')
           .configure_legend(symbolSize=markerSize, symbolOpacity=markerOpacity, symbolStrokeWidth=2.2, symbolStrokeColor='black', 
                             titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                            strokeColor='gray',padding=10,cornerRadius=10,
                            labelLimit = 1000 # Let legend labels be as long as they want
                            )
    )

    return layered

In [5]:
for subtype in titers.subtype.unique():
    data = titers[titers['subtype'].isin([subtype])]
    data = data[data['timepoint'].isin(['d0', 'd29', 'd121 or d181'])]
    plot = plot_titers_vaccination_cohorts(data, sort_order = ['Fluarix (n=15)', 'mRNA-1010'], 
                                           _range=[30, 18000], title = f'{subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_pre-post_titers.pdf')
    plot.save(outfile, dpi = 600)


In [6]:
IFrame(os.path.join(resultsdir, f'H3N2_pre-post_titers.pdf'), width=700, height=500)

In [7]:
IFrame(os.path.join(resultsdir, f'H1N1_pre-post_titers.pdf'), width=700, height=500)

# Plot log fold-change titers for each group from day 0 to day 29 and day 0 to day 121/181

In [8]:
# Pivot the table to have timepoints as columns
pivot_titers = titers.pivot_table(
    index=['participant', 'vaccine', 'virus', 'subtype', ],
    columns='timepoint',
    values='titer'
)

# Calculate fold-changes
# pivot_titers['fold_change_d29'] = pivot_titers['d29'] / pivot_titers['d0']
# if 'd121' in pivot_titers.columns:
#     pivot_titers['fold_change_d121'] = pivot_titers['d121'] / pivot_titers['d0']
# pivot_titers['fold_change_d181'] = pivot_titers['d181'] / pivot_titers['d0']

# Only compute fold-change if both values are present, otherwise get NaN
pivot_titers['fold_change_d29'] = np.where(
    pivot_titers['d0'].notna() & pivot_titers['d29'].notna(),
    pivot_titers['d29'] / pivot_titers['d0'],
    np.nan
)

# Repeat for d121 if it exists
if 'd121' in pivot_titers.columns:
    pivot_titers['fold_change_d121'] = np.where(
        pivot_titers['d0'].notna() & pivot_titers['d121'].notna(),
        pivot_titers['d121'] / pivot_titers['d0'],
        np.nan
    )

pivot_titers['fold_change_d181'] = np.where(
    pivot_titers['d0'].notna() & pivot_titers['d181'].notna(),
    pivot_titers['d181'] / pivot_titers['d0'],
    np.nan
)




# There's inconsistency between the last date being d121 or d181, so making a column that has both
if 'd121' in pivot_titers.columns:
    pivot_titers['fold_change_d121_or_d181'] = pivot_titers.get('d121', pd.NA).combine_first(pivot_titers.get('d181', pd.NA)) / pivot_titers['d0']

# Compute log2 fold changes
pivot_titers['log2_fc_d29'] = np.log2(pivot_titers['fold_change_d29'])
pivot_titers['log2_fc_d181'] = np.log2(pivot_titers['fold_change_d181'])
if 'd121' in pivot_titers.columns:
    pivot_titers['log2_fc_d121'] = np.log2(pivot_titers['fold_change_d121'])
    pivot_titers['log2_fc_d121_or_d181'] = np.log2(pivot_titers['fold_change_d121_or_d181'])

pivot_titers = pivot_titers.reset_index()

In [9]:
def plot_FC_vaccination_cohorts(data, y_axis, _range = [0, 1], title=None):
    # Make plot with all individuals and median dots
    color_scheme = alt.Color('vaccine', sort=['Fluarix (n=15)']).scale(range=color_palette)
    titer_range = _range
    titleFontSize=18
    labelFontSize=18
    lineOpacity = 0.2
    lineSize = 2.8
    markerOpacity = 0.8
    markerSize = 160

    band = (alt.Chart(data)
            .mark_errorband(extent='iqr', opacity=0.4)
            .encode(alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right'),             
                          sort = virus_order),
                    alt.Y(f'{y_axis}', 
                          scale =alt.Scale(domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title=["log2FC titer", "from pre-vaccination"])),
                color = color_scheme,)
           ) 
    
    points = (alt.Chart(data)
              .mark_point(size = markerSize, stroke = 'black', strokeWidth = 2.2, filled=True,  opacity=markerOpacity)
              .encode(alt.X('virus', sort = virus_order),
                      alt.Y(f'median({y_axis})'),
                      color = color_scheme,))
        
    layered = (alt.layer(band, points)
               .properties(title=title)
               .configure_header(title=None,
                                  labelFontSize=labelFontSize,labelFontWeight='bold',
                                  labelOrient='right', 
                                )
               .configure_title(align='center', anchor='middle', fontSize=titleFontSize, fontWeight='bold')
           .configure_legend(symbolSize=markerSize, symbolOpacity=markerOpacity, symbolStrokeWidth=2.2, symbolStrokeColor='black', 
                             titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                            strokeColor='gray',padding=10,cornerRadius=10,
                            labelLimit = 1000 # Let legend labels be as long as they want
                            )
    )

    return layered

## Log FC in neutralization titers at day 29

In [10]:
for subtype in titers.subtype.unique():
    data = pivot_titers[pivot_titers['subtype'].isin([subtype])].dropna(subset=['log2_fc_d29'])

    plot = plot_FC_vaccination_cohorts(data, 
                                           y_axis = 'log2_fc_d29',
                                           _range=[-0.5, 4.5], title = ['Log2FC in titers from day 0 to day 29 for', f'{subtype} 2023-2024 circulating strains and recent vaccine strains'])
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_day29_pre-post_fold-change.pdf')
    plot.save(outfile, dpi = 600)


In [11]:
IFrame(os.path.join(resultsdir, 'H3N2_day29_pre-post_fold-change.pdf'), width=700, height=500)

In [12]:
IFrame(os.path.join(resultsdir, 'H1N1_day29_pre-post_fold-change.pdf'), width=700, height=500)

## Log FC in neutralization titers at day 121 or 181

In [13]:
if 'd121' in pivot_titers.columns:
    y_axis = 'log2_fc_d121_or_d181'
else:
    y_axis = 'log2_fc_d181'

for subtype in titers.subtype.unique():
    data = pivot_titers[pivot_titers['subtype'].isin([subtype])].dropna(subset=[y_axis])

    plot = plot_FC_vaccination_cohorts(data, 
                                       y_axis = y_axis,
                                       _range = [-1, 3],
                                       title = ['Log2FC in titers from day 0 to day 121 or 181 for', f'{subtype} 2023-2024 circulating strains and recent vaccine strains'])
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_day121-181_pre-post_fold-change.pdf')
    plot.save(outfile, dpi = 600)


In [14]:
IFrame(os.path.join(resultsdir, 'H3N2_day121-181_pre-post_fold-change.pdf'), width=700, height=500)

In [15]:
IFrame(os.path.join(resultsdir, 'H1N1_day121-181_pre-post_fold-change.pdf'), width=700, height=500)

## Statistical analysis of median log2FC values

In [31]:
pivot_titers

timepoint,participant,vaccine,virus,subtype,d0,d121,d121 or d181,d181,d29,fold_change_d29,fold_change_d121,fold_change_d181,fold_change_d121_or_d181,log2_fc_d29,log2_fc_d181,log2_fc_d121,log2_fc_d121_or_d181
0,397-001,Fluarix (n=14),A/AbuDhabi/6753/2023_H3N2,H3N2,90.44,NaN,107.20,107.20,245.0,2.708978,NaN,1.185316,1.185316,1.437749,0.245272,NaN,0.245272
1,397-001,Fluarix (n=14),A/Arizona/IVYG43Q65T3/2023_H1N1,H1N1,74.66,NaN,107.40,107.40,115.3,1.544334,NaN,1.438521,1.438521,0.626985,0.524587,NaN,0.524587
2,397-001,Fluarix (n=14),A/Bangkok/P3599/2023_H3N2,H3N2,66.68,NaN,110.40,110.40,208.6,3.128374,NaN,1.655669,1.655669,1.645413,0.727414,NaN,0.727414
3,397-001,Fluarix (n=14),A/Bangkok/P3755/2023_H3N2,H3N2,66.08,NaN,74.95,74.95,385.9,5.839891,NaN,1.134231,1.134231,2.545941,0.181715,NaN,0.181715
4,397-001,Fluarix (n=14),A/Bhutan/0006/2023_H3N2,H3N2,84.48,NaN,106.40,106.40,559.4,6.621686,NaN,1.259470,1.259470,2.727199,0.332816,NaN,0.332816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2585,397-032,Fluarix (n=14),A/Victoria/4897/2022_IVR-238_H1N1,H1N1,1419.00,NaN,1000.00,1000.00,1419.0,1.000000,NaN,0.704722,0.704722,0.000000,-0.504875,NaN,-0.504875
2586,397-032,Fluarix (n=14),A/Wisconsin/27/2023_H3N2,H3N2,190.40,NaN,588.90,588.90,3188.0,16.743697,NaN,3.092962,3.092962,4.065546,1.628989,NaN,1.628989
2587,397-032,Fluarix (n=14),A/Wisconsin/588/2019_H1N1,H1N1,752.00,NaN,645.80,645.80,824.1,1.095878,NaN,0.858777,0.858777,0.132087,-0.219645,NaN,-0.219645
2588,397-032,Fluarix (n=14),A/Wisconsin/67/2022_H1N1,H1N1,113.50,NaN,118.30,118.30,282.0,2.484581,NaN,1.042291,1.042291,1.313003,0.059758,NaN,0.059758


In [33]:
# Mann Whitney U two-sided test
p_values = []
median_diff = []
virus_with_stats = []

for titer, titer_df in pivot_titers.groupby('virus'):
    groupA = titer_df[titer_df['vaccine'] == 'Fluarix (n=14)']['log2_fc_d29'].dropna()
    groupB = titer_df[titer_df['vaccine'] == 'mRNA-1010 (n=13)']['log2_fc_d29'].dropna()

    if len(groupA) > 0 and len(groupB) > 0:
        # Mann-Whitney U Test (two-sided)
        stat, p = mannwhitneyu(groupA, groupB, alternative='two-sided')
        p_values.append(p)
        median_diff.append(groupA.median() - groupB.median())
        virus_with_stats.append(titer)
    else:
        # Not enough data — skip
        continue

# # FDR correction
# reject, pvals_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')


# Convert to array for further processing
pvals = np.array(p_values)
m = len(pvals)
alpha = 0.05

# Benjamini-Hochberg FDR thresholding
sorted_indices = np.argsort(pvals)
sorted_pvals = pvals[sorted_indices]
bh_thresholds = (np.arange(1, m + 1) / m) * alpha

# Find largest p ≤ threshold
significant = sorted_pvals <= bh_thresholds
if np.any(significant):
    max_idx = np.where(significant)[0].max()
    cutoff_pval = sorted_pvals[max_idx]
    fdr_significant_mask = pvals <= cutoff_pval
else:
    fdr_significant_mask = np.full_like(pvals, False, dtype=bool)



# Summary results
results_df = pd.DataFrame({
    'virus': virus_with_stats,
    'median(Fluarix)-median(mRNA)': median_diff,
    'p_value': p_values,
    'FDR_significant': fdr_significant_mask
    # 'p_value_FDR': pvals_corrected,
    # 'significant_FDR': reject
}).sort_values('p_value')

results_df.reset_index(drop=True)

,virus,median(Fluarix)-median(mRNA),p_value,FDR_significant
0,A/Victoria/1389/2023_H1N1,-1.389009,0.002481,False
1,A/Netherlands/1739/2023_H1N1,-1.317769,0.003327,False
2,A/Wisconsin/588/2019_H1N1,-1.607127,0.006112,False
3,A/Wisconsin/67/2022_H1N1,-1.374360,0.007077,False
4,A/Norway/7787/2023_H1N1,-1.167126,0.010846,False
...,...,...,...,...
91,A/Finland/399/2023_H3N2,-0.552825,0.816961,False
92,A/Thailand/8/2022_H3N2,-0.316751,0.827143,False
93,A/KANAGAWA/AC2316/2023_H3N2,-0.159105,0.877731,False
94,A/SouthAfrica/R06359/2023_H3N2,0.091419,0.897685,False


In [34]:
results_df.reset_index(drop=True).query('p_value <= 0.05')

,virus,median(Fluarix)-median(mRNA),p_value,FDR_significant
0,A/Victoria/1389/2023_H1N1,-1.389009,0.002481,False
1,A/Netherlands/1739/2023_H1N1,-1.317769,0.003327,False
2,A/Wisconsin/588/2019_H1N1,-1.607127,0.006112,False
3,A/Wisconsin/67/2022_H1N1,-1.374360,0.007077,False
4,A/Norway/7787/2023_H1N1,-1.167126,0.010846,False
5,A/Pennsylvania/51/2023_H1N1,-1.126399,0.012451,False
6,A/HongKong/2671/2019_H3N2,-1.569414,0.016238,False
7,A/Jeonbuk/1544/2023_H1N1,-1.131409,0.016304,False
8,A/Victoria/4897/2022_IVR-238_H1N1,-1.381151,0.016304,False
9,A/Texas/50/2012_H3N2,-0.955974,0.019270,False


## Distributions

In [18]:
single_virus_df = pivot_titers.query('virus == "A/AbuDhabi/6753/2023_H3N2"').reset_index(drop=True)

for vax in single_virus_df.vaccine.unique():
    print(vax)

    print('there are this many participants', (len(single_virus_df.query(f'vaccine == "{vax}"'))))
    print('this many have day 121', (len(single_virus_df.query(f'vaccine == "{vax}"')[['d121']].dropna())))

Fluarix (n=14)
there are this many participants 14
this many have day 121 4
mRNA-1010 (n=13)
there are this many participants 13
this many have day 121 2


In [19]:
pivot_titers.query('subtype == "H1N1"')

alt.Chart(pivot_titers.query('subtype == "H1N1"')).mark_bar().encode(
    alt.X('d0:Q', bin=True),  # bin=True automatically chooses bin size
    y='count()'
).properties(
    title='Distribution of values'
)

alt.Chart(...)

## Show with lines

In [20]:
def plot_FC_vaccination_cohorts_with_lines(data, y_axis, _range = [0, 1], title=None):
    # Make plot with all individuals and median dots
    color_scheme = alt.Color('vaccine', sort=['Fluarix (n=15)']).scale(range=color_palette)
    titer_range = _range
    titleFontSize=18
    labelFontSize=18
    lineOpacity = 0.2
    lineSize = 2.8
    markerOpacity = 0.8
    markerSize = 160

    band = (alt.Chart(data)
            .mark_errorband(extent='iqr', opacity=0.4)
            .encode(alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right'),             
                          sort = virus_order),
                    alt.Y(f'{y_axis}', 
                          scale =alt.Scale(domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title=["log2FC titer", "from pre-vaccination"])),
                color = color_scheme,)
           ) 


    line = (alt.Chart(data)
            .mark_line(size = lineSize, point = False, opacity = lineOpacity)
            .encode(
                alt.X('virus', axis = alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize,
                                          title = None,labelLimit = 1000, labelAlign = 'right'),             
                          sort = virus_order),
                alt.Y(f'{y_axis}', 
                          scale =alt.Scale(domain=_range, nice=False), 
                          axis=alt.Axis(grid=False, titleFontSize=titleFontSize, labelFontSize=labelFontSize, title=["log2FC titer", "from pre-vaccination"])),
                detail = 'participant',
                color = color_scheme,
                # order = 'virus_order'
            )
           )
    
    points = (alt.Chart(data)
              .mark_point(size = markerSize, stroke = 'black', strokeWidth = 2.2, filled=True,  opacity=markerOpacity)
              .encode(alt.X('virus', sort = virus_order),
                      alt.Y(f'median({y_axis})'),
                      color = color_scheme,))
        
    layered = (alt.layer(band, line, points)
               .properties(title=title)
               .configure_header(title=None,
                                  labelFontSize=labelFontSize,labelFontWeight='bold',
                                  labelOrient='right', 
                                )
               .configure_title(align='center', anchor='middle', fontSize=titleFontSize, fontWeight='bold')
           .configure_legend(symbolSize=markerSize, symbolOpacity=markerOpacity, symbolStrokeWidth=2.2, symbolStrokeColor='black', 
                             titleFontSize=titleFontSize, labelFontSize = labelFontSize,
                            strokeColor='gray',padding=10,cornerRadius=10,
                            labelLimit = 1000 # Let legend labels be as long as they want
                            )
    )

    return layered

In [21]:
for subtype in titers.subtype.unique():
    data = pivot_titers[pivot_titers['subtype'].isin([subtype])]

    plot = plot_FC_vaccination_cohorts_with_lines(data, 
                                           y_axis = 'log2_fc_d29',
                                           _range=[-0.6, 6], title = f'Log2 FC day 29 titers to {subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_day29_pre-post_fold-change_LINES.pdf')
    plot.save(outfile, dpi = 600)


In [22]:
IFrame(os.path.join(resultsdir, 'H3N2_day29_pre-post_fold-change_LINES.pdf'), width=700, height=500)

In [23]:
IFrame(os.path.join(resultsdir, 'H1N1_day29_pre-post_fold-change_LINES.pdf'), width=700, height=500)

In [24]:
if 'd121' in pivot_titers.columns:
    y_axis = 'log2_fc_d121_or_d181'
else:
    y_axis = 'log2_fc_d181'

for subtype in titers.subtype.unique():
    data = pivot_titers[pivot_titers['subtype'].isin([subtype])]

    plot = plot_FC_vaccination_cohorts_with_lines(data, 
                                           y_axis = y_axis,
                                           _range=[-1.5, 4.5], title = f'Log2 FC day 121/181 titers to {subtype} 2023-2024 circulating strains and recent vaccine strains')
    # Save final plot
    outfile = os.path.join(resultsdir, f'{subtype}_day121-181_pre-post_fold-change_LINES.pdf')
    plot.save(outfile, dpi = 600)


In [25]:
IFrame(os.path.join(resultsdir, 'H3N2_day121-181_pre-post_fold-change_LINES.pdf'), width=700, height=500)

In [26]:
IFrame(os.path.join(resultsdir, 'H1N1_day121-181_pre-post_fold-change_LINES.pdf'), width=700, height=500)